In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:8], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [4]:
print(chars)
print(stoi)
print(itos)

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
X = []
Y = []

block_size = 3

for w in words:
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

In [6]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [7]:
# print(X)
# print(Y)
print(X.shape)
print(Y.shape)

torch.Size([228146, 3])
torch.Size([228146])


In [8]:
C = torch.randn(27,10)
C.shape
print(X[0])
print(X[0].shape)

emb = C[X[0]]

print(emb.shape)

tensor([0, 0, 0])
torch.Size([3])
torch.Size([3, 10])


In [9]:
emb = C[X[2]]

print("X[2]:", X[2])
print("emb shape:", emb.shape)

emb_flat = emb.view(-1, 30)

print("flattened shape:", emb_flat.shape)

X[2]: tensor([ 0,  5, 13])
emb shape: torch.Size([3, 10])
flattened shape: torch.Size([1, 30])


In [15]:
W1 = torch.randn(30, 200)
z = emb_flat @ W1
print("z:", z.shape)

b1 = torch.randn(200)
z = z + b1

print("z after bias:", z.shape)

h = torch.tanh(z)

print("h:", h.shape)

z: torch.Size([1, 200])
z after bias: torch.Size([1, 200])
h: torch.Size([1, 200])


In [17]:
W2 = torch.randn((200, 27))
b2 = torch.randn(27)

In [20]:
logits = h @ W2 + b2
logits.shape, logits

(torch.Size([1, 27]),
 tensor([[-25.4210,  -2.1195,  -2.6902,  10.9239,  -1.5417,   1.2274,  11.8292,
          -20.4797,   8.1457, -27.7454,  10.5824, -12.0261,  11.4261,   9.5798,
           -3.2100,   7.2781,  21.7760,  -8.6079,  -0.4667,  -2.2850,  -4.6886,
          -12.2389, -23.6012,  -1.9905,   4.6514,  13.5281, -14.8565]]))

In [21]:
counts = logits.exp()
counts.shape, counts

(torch.Size([1, 27]),
 tensor([[9.1156e-12, 1.2009e-01, 6.7869e-02, 5.5488e+04, 2.1402e-01, 3.4123e+00,
          1.3720e+05, 1.2758e-09, 3.4486e+03, 8.9188e-13, 3.9436e+04, 5.9861e-06,
          9.1684e+04, 1.4470e+04, 4.0356e-02, 1.4482e+03, 2.8654e+09, 1.8265e-04,
          6.2709e-01, 1.0177e-01, 9.1993e-03, 4.8384e-06, 5.6248e-11, 1.3662e-01,
          1.0473e+02, 7.5020e+05, 3.5309e-07]]))

In [22]:
prob = counts / counts.sum(1, keepdims=True)  # why do we need, keepdims here
prob.shape, prob

(torch.Size([1, 27]),
 tensor([[3.1800e-21, 4.1895e-11, 2.3676e-11, 1.9358e-05, 7.4661e-11, 1.1904e-09,
          4.7863e-05, 4.4506e-19, 1.2031e-06, 3.1114e-22, 1.3758e-05, 2.0883e-15,
          3.1985e-05, 5.0480e-06, 1.4079e-11, 5.0521e-07, 9.9962e-01, 6.3718e-14,
          2.1877e-10, 3.5505e-11, 3.2093e-12, 1.6879e-15, 1.9623e-20, 4.7662e-11,
          3.6536e-08, 2.6171e-04, 1.2318e-16]]))